In [ ]:
import pandas as pd
import numpy as np
from astropy.stats import sigma_clip
from scipy.stats import binned_statistic
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import partial_dependence, permutation_importance

# Helper Function That Plots The Scatter with Error Bars #

Identical to the RF notebook — see that notebook for the full derivation.

In [ ]:
nbins = 10

def scatter_with_errors(ax, xcol, nbins=None):
    x = df[xcol].values
    y = df["residual_centered"].values
    yerr = yerr_all.values

    ax.scatter(x, y, **point_kwargs)

    ax.errorbar(
        x, y, yerr=yerr,
        fmt="none",
        ecolor="black",
        elinewidth=0.5,
        alpha=0.25,
        zorder=2,
    )

    ax.axhline(0, linestyle="--", color="orange")

    _, bin_edges, binnumber = binned_statistic(x, x, statistic="count", bins=nbins)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    binned_y    = []
    binned_yerr = []

    for i in range(1, len(bin_edges)):
        in_bin = binnumber == i
        if np.sum(in_bin) == 0:
            binned_y.append(np.nan)
            binned_yerr.append(np.nan)
            continue
        y_bin   = y[in_bin]
        binned_y.append(np.mean(y_bin))
        binned_yerr.append(np.std(y_bin, ddof=1))

    binned_y    = np.array(binned_y)
    binned_yerr = np.array(binned_yerr)
    ok = np.isfinite(binned_y) & np.isfinite(binned_yerr)

    ax.errorbar(
        bin_centers[ok],
        binned_y[ok],
        yerr=binned_yerr[ok],
        fmt="o",
        color="blue",
        markersize=7,
        capsize=3,
        zorder=5,
    )

# Data Overview #

In [ ]:
df = pd.read_csv("ZTF_DESI_data/ZTF_resid_cent_hostprop_no_x1_c.csv")

print(len(df), "SNe before paper-based quality cuts.")
df = df[(df["lccoverage_flag"] == 1) & (df["fitquality_flag"] == 1)]
print(len(df), "SNe after paper-based quality cuts.")

df["SDSS_g_minus_r"] = df["ABSMAG01_SDSS_G"] - df["ABSMAG01_SDSS_R"]

resid_before = df["residual_centered"]
clip = sigma_clip(df["residual_centered"], sigma=3, maxiters=1)
df = df.loc[~clip.mask].reset_index(drop=True)

resid_bins = np.linspace(-1, 2, 50)
plt.hist(resid_before, bins=resid_bins, label="Before sigma clip", color="black")
plt.hist(df["residual_centered"], bins=resid_bins, label="After sigma clip", color="gray")
plt.xlabel(r"$\Delta \mu$")
plt.ylabel("Count")
plt.legend()
plt.show()
print(len(df), "SNe after sigma clipping (Δμ)")

mask_sfr = df["SFR"] <= 2.5
df = df.loc[mask_sfr].reset_index(drop=True)
print(len(df), "SNe after SFR <= 2.5 cut.")

mask_dn4000 = df["DN4000"] >= 0.5
df = df.loc[mask_dn4000].reset_index(drop=True)
print(len(df), "SNe after DN4000 >= 0.5 cut.")

mask_age = df["AGE"] >= 2
df = df.loc[mask_age].reset_index(drop=True)
print(len(df), "SNe after AGE >= 2 Gyr cut.")

plt.figure()
plt.hist(df["redshift"], bins=30, color="steelblue", alpha=0.7)
plt.xlabel("redshift")
plt.ylabel("Count")
plt.title("Redshift Histogram")
plt.show()

yerr_all = df["sigma_mu_meas"]

# --- 2x4 overview grid ---
nbins = 10
fig, axes = plt.subplots(2, 4, figsize=(14, 8), sharey=True)
axes = axes.flatten()
point_kwargs = dict(alpha=0.2, s=50, color="grey", zorder=3)

for ax, col, label in zip(
    axes,
    ["LOGMSTAR", "SFR", "VDISP", "DN4000", "SDSS_g_minus_r", "AGE", "x1", "c"],
    [
        r"$\log_{10}(M_\star / M_{\odot})$",
        r"${\rm SFR}\ (M_{\odot}/yr)$",
        r"${\rm VDisp}\ (km/s)$",
        r"${\rm Dn4000}$",
        r"$g - r\ {\rm (SDSS)}$",
        r"${\rm AGE}\ [{\rm Gyr}]$",
        "x1",
        "c",
    ],
):
    scatter_with_errors(ax, col, nbins=nbins)
    ax.set_xlabel(label)

axes[0].set_ylabel(r"$\Delta \mu$")
axes[4].set_ylabel(r"$\Delta \mu$")
fig.tight_layout()
plt.show()

# Skopt Best-Found MLP — Fixed Hyperparameters #

These are the hyperparameters selected by `BayesSearchCV` (skopt) from
`mlp_skopt_bayesian.ipynb`.  We hard-code them here so the results notebook
is fully reproducible without re-running the search.

| Parameter | Value |
|---|---|
| `hidden_layer_sizes` | (64, 32, 16) |
| `activation` | tanh |
| `solver` | adam |
| `alpha` | 0.4489 |
| `learning_rate_init` | 0.005996 |
| `max_iter` | 2000 |
| `early_stopping` | False |

**Note on sample weights:** `MLPRegressor` does not support `sample_weight`
in `fit()`, unlike `RandomForestRegressor`.  The network is trained on
unweighted residuals; the measurement errors `sigma_mu_meas` are used only
in the diagnostic plots.

**Note on partial dependence:** `method='recursion'` is only available for
tree-based models.  We use `method='brute'` here, which is the literal
definition — see the RF notebook for the derivation.

In [ ]:
# =====================================================
# SKOPT BEST MLP — TRAINING BLOCK
# =====================================================

feature_cols = [
    "LOGMSTAR",
    "c",
    "x1",
    "SFR",
    "VDISP",
    "DN4000",
    "SDSS_g_minus_r",
    "AGE",
    "redshift",
]
target_col = "residual_centered"

X = df[feature_cols]
y = df[target_col]

Xtr, Xte, ytr, yte, yerr_tr, yerr_te = train_test_split(
    X, y, yerr_all, test_size=0.2, random_state=42
)

# MLP requires feature scaling — wrap in a Pipeline
mlp_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(
        hidden_layer_sizes=(64, 32, 16),
        activation="tanh",
        solver="adam",
        alpha=0.4489379023123383,
        learning_rate_init=0.005995964127756074,
        max_iter=2000,
        early_stopping=False,
        random_state=42,
    )),
])

# MLPRegressor does not accept sample_weight — train unweighted
mlp_pipe.fit(Xtr, ytr)
yhat = mlp_pipe.predict(Xte)

# --- Performance metrics ---
ytr_pred  = mlp_pipe.predict(Xtr)
test_mae  = mean_absolute_error(yte, yhat)
test_rmse = float(np.sqrt(mean_squared_error(yte, yhat)))
test_r2   = r2_score(yte, yhat)
train_r2  = r2_score(ytr, ytr_pred)
test_nmad = 1.4826 * float(np.median(np.abs(np.asarray(yhat) - np.asarray(yte))))

print("=" * 45)
print("  SKOPT MLP — TEST-SET RESULTS")
print("=" * 45)
print(f"  Test MAE   : {test_mae:.4f} mag")
print(f"  Test RMSE  : {test_rmse:.4f} mag")
print(f"  Test NMAD  : {test_nmad:.4f} mag")
print(f"  R² (test)  : {test_r2:.4f}")
print(f"  R² (train) : {train_r2:.4f}")
print(f"  ΔR² overfit: {train_r2 - test_r2:.4f}")
print("=" * 45)

# --- Permutation feature importance (MLP has no built-in feature_importances_) ---
perm = permutation_importance(
    mlp_pipe, Xte, yte,
    n_repeats=30,
    random_state=42,
    scoring="neg_mean_absolute_error",
)
fi = pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False)
print("\nPermutation Feature Importances (mean decrease in MAE):")
print(fi)

In [ ]:
# --- Compare distributions: raw residuals vs MLP predictions ---
y_all = df[target_col].to_numpy()
bins  = np.linspace(-1.5, 1.5, 30)

plt.hist(yhat,  bins=bins, alpha=0.5, color="orange", label="MLP Predictions (test set)")
plt.hist(y_all, bins=bins, alpha=0.3, color="grey",   label="Raw Data (full sample)")
plt.xlabel(r"$\Delta \mu$")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

## Partial Dependence Plots

Same interpretation as the RF notebook.  Because the MLP is not a tree-based
model, `method='brute'` is used: for each grid value $x_j$ of the target
feature, the function sets that feature to $x_j$ for every training sample,
runs prediction, and averages:

$$
f_{\rm PD}(x_j) = \frac{1}{N}\sum_{i=1}^{N} f\!\left(x_j,\, \mathbf{x}_{-j,i}\right)
$$

The `StandardScaler` inside the pipeline is applied automatically — we pass
the *raw* (unscaled) feature matrix to `partial_dependence`, and sklearn
routes it through the full pipeline.

In [ ]:
def plot_pd_panel(ax, feature, pipe, X, feature_cols, nbins=10):
    X_pd = X[feature_cols]
    X_pd = X_pd[X_pd.redshift <= 0.06]

    res    = partial_dependence(pipe, X_pd, features=[feature], method="brute")
    x_grid = res["grid_values"][0]
    y_grid = res["average"][0]

    ax.scatter(x_grid, y_grid, alpha=0.6, s=30, color="orange",
               label="PD (marginalized)", zorder=4)

    global point_kwargs
    point_kwargs = {**point_kwargs, "label": "Raw Data"}
    scatter_with_errors(ax, feature, nbins=nbins)
    ax.errorbar([], [], fmt="o", color="blue", markersize=4, label="Binned Raw data")

    ax.set_xlabel(feature)
    ax.set_ylabel(r"$\Delta \mu$")


fig, axes = plt.subplots(3, 3, figsize=(16, 12), sharey=True)

# Row 1
plot_pd_panel(axes[0, 0], "LOGMSTAR",      mlp_pipe, X, feature_cols, nbins=nbins)

plot_pd_panel(axes[0, 1], "c",             mlp_pipe, X, feature_cols, nbins=nbins)
coef_c = np.polyfit(df["c"], y, 1)
x_fit_c = np.linspace(df["c"].min(), df["c"].max(), 200)
axes[0, 1].plot(x_fit_c, np.polyval(coef_c, x_fit_c), color="blue", linewidth=2)
print("Linear fit coefficients for c (slope, intercept):", coef_c)

mask_c = (df["c"] >= -0.2) & (df["c"] <= 0.3)
coef_c_limited = np.polyfit(df.loc[mask_c, "c"], y.values[mask_c], 1)
x_fit_c_limited = np.linspace(-0.2, 0.3, 200)
axes[0, 1].plot(x_fit_c_limited, np.polyval(coef_c_limited, x_fit_c_limited),
                color="green", linewidth=2)
print("Linear fit for c (limited range, slope, intercept):", coef_c_limited)

y_predict  = mlp_pipe.predict(X[feature_cols])
coef_c_mlp = np.polyfit(df.loc[mask_c, "c"], y_predict[mask_c], 1)
axes[0, 1].plot(x_fit_c_limited, np.polyval(coef_c_mlp, x_fit_c_limited),
                color="red", linewidth=2)
print("Linear fit for c (MLP prediction, slope, intercept):", coef_c_mlp)

plot_pd_panel(axes[0, 2], "x1",            mlp_pipe, X, feature_cols, nbins=nbins)
coef_x1 = np.polyfit(df["x1"], y, 1)
x_fit_x1 = np.linspace(df["x1"].min(), df["x1"].max(), 200)
axes[0, 2].plot(x_fit_x1, np.polyval(coef_x1, x_fit_x1), color="blue", linewidth=2)
print("Linear fit for x1 (slope, intercept):", coef_x1)

# Row 2
plot_pd_panel(axes[1, 0], "SFR",           mlp_pipe, X, feature_cols, nbins=nbins)
plot_pd_panel(axes[1, 1], "VDISP",         mlp_pipe, X, feature_cols, nbins=nbins)
plot_pd_panel(axes[1, 2], "DN4000",        mlp_pipe, X, feature_cols, nbins=nbins)

# Row 3
plot_pd_panel(axes[2, 0], "SDSS_g_minus_r",mlp_pipe, X, feature_cols, nbins=nbins)
plot_pd_panel(axes[2, 1], "AGE",           mlp_pipe, X, feature_cols, nbins=nbins)
plot_pd_panel(axes[2, 2], "redshift",      mlp_pipe, X, feature_cols, nbins=20)
axes[2, 2].set_xlim(0, 0.1)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

mu_bins = np.linspace(-1.5, 1.5, 30)
plt.hist(y_predict - y, bins=mu_bins, alpha=0.5, color="orange",
         label="MLP residuals (prediction − truth)")
plt.xlabel(r"$\hat{y} - y$")
plt.ylabel("Count")
plt.legend()
plt.show()
print("Std of (MLP prediction − truth):", np.std(y_predict - y))